In [1]:
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, roc_curve
from scipy.stats import spearmanr, pearsonr
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import pickle
import itertools

In [2]:
AA_LIST = list("ACDEFGHIKLMNPQRSTVWYO-")
N_AA = len(AA_LIST)
aa_to_idx = {aa: i for i, aa in enumerate(AA_LIST)}
idx_to_aa = {v: k for k, v in aa_to_idx.items()}

def min_max_norm(data):
    data_min = np.min(data)
    data_max = np.max(data)
    normalized_data = (data - data_min) / (data_max - data_min)
    return normalized_data

def clipping_max_norm(data, cutoff_val=None, cutoff_percentile=99, return_cutoff=False):
    if cutoff_val is None:
        upper_limit = np.percentile(data, cutoff_percentile)
    else:
        upper_limit = cutoff_val
    clipped_matrix = np.clip(data, a_min=0, a_max=upper_limit)
    normalized_data = clipped_matrix / upper_limit
    if return_cutoff:
        return normalized_data, upper_limit
    else:
        return normalized_data

def process_single_attention(file_path):
    attn_all_layers = np.load(file_path)
    return get_res_talk(attn_all_layers)

def get_res_talk(attn_all_layers):
    aa_attn = attn_all_layers[:, :, 1:, 1:]
    cls_attn = attn_all_layers[:, :, 0, 1:]
    # Calculate min and max along the last axis (sequence length) for all layers/heads simultaneously
    c_min = cls_attn.min(axis=-1, keepdims=True)
    c_max = cls_attn.max(axis=-1, keepdims=True)
    norm_cls = (cls_attn - c_min) / (c_max - c_min)
    # 'lhij' represents dimensions: layers(l), heads(h), seq_i(i), seq_j(j)
    # 'lhi' represents norm_cls dimensions.
    # '->ij' tells NumPy to multiply them and sum over 'l' and 'h', outputting shape (i, j).
    sum_weighted_attn = np.einsum('lhij,lhi->ij', aa_attn, norm_cls)
    return sum_weighted_attn

def mutate(sequence, mutations, residue_index):
    current_sequence = sequence.copy()
    res_to_idx = {res: i for i, res in enumerate(residue_index)}
    for reslabel, original_aa, mutated_aa in mutations:
        res_pos = res_to_idx[reslabel]
        if current_sequence[res_pos] != original_aa:
            raise ValueError(f"Expected {original_aa} at {reslabel}, got {current_sequence[res_pos]}")
        current_sequence[res_pos] = mutated_aa
    return current_sequence

def predict_E(sequence, communication_map, h_map, epistatic_map, residue_index,
              coupling=True, return_map=False, pval_cutoff=1.0, count_cutoff=0):
    L = len(sequence)
    H = np.zeros(L)
    C = np.zeros(L)
    res_seq = list(zip(residue_index, sequence))
    for target_idx, (target_reslabel, target_aa) in enumerate(res_seq):
        # Get the Baseline (H)
        H[target_idx] = h_map.get((target_reslabel, target_aa), 0.0)
        w_row = communication_map[target_idx]
        c_val = 0.0
        for src_idx, (src_reslabel, src_aa) in enumerate(res_seq):
            w = w_row[src_idx]
            ep_data = epistatic_map.get((target_reslabel, target_aa, src_reslabel, src_aa))
            if ep_data is not None:
                c, _, p_val, s_count = ep_data
                if p_val < pval_cutoff and s_count > count_cutoff:
                    c_val += (w * c) 
        C[target_idx] = c_val
    if coupling:
        E = H + C
    else:
        E = H
    sum_E = E.sum()
    if return_map:
        return sum_E, H, C
    return sum_E

In [20]:
AMP_df = pd.read_csv("../data/AMP_variants.csv")
AMP_seqno_array = AMP_df["Seq_no"].values
AMP_aligned_sequences = np.array([list(seq) for seq in AMP_df['Sequence']])
AMP_pic50 = AMP_df["pIC50"].values

resno_df = pd.read_csv("../data/selected_residues_IC80.csv")
resno_array = resno_df["ResLabel"].values

data_dir = '/pl/active/rdi_data/fahsai/PLM'
with open(data_dir+'/results/coupling/epistatic_map.pkl', 'rb') as c:
    loaded_data = pickle.load(c)
    epistatic_map = loaded_data['epistatic_map']
with open(data_dir+'/results/coupling/h_map.pkl', 'rb') as c:
    h_map = pickle.load(c)
with open(data_dir+'/results/full/attn_cutoff.pkl', 'rb') as c:
    saved_model_cutoffs = pickle.load(c)

N_SEQ, L_SEQ = AMP_aligned_sequences.shape
selected_models = ["rep_5", "rep_6", "rep_7", "rep_9", "rep_10"]

In [21]:
attn_dir = "/scratch/alpine/fana6609/ML/PLM-Epistasis/results/AMP/attention/variants"

residue_com_map = np.zeros([len(selected_models), N_SEQ, L_SEQ, L_SEQ])
for n, model_name in enumerate(selected_models):
    model_sum_attn_attr = np.zeros([N_SEQ, L_SEQ, L_SEQ])
    attn_file_paths = [os.path.join(attn_dir, model_name, f"{no}_attentions_raw.npy") for no in AMP_seqno_array]

    with ThreadPoolExecutor(max_workers=8) as executor: 
        attn_results = list(tqdm(executor.map(process_single_attention, attn_file_paths), total=N_SEQ))
    
    for i, attn_res in enumerate(attn_results):
        model_sum_attn_attr[i] = attn_res
    
    normalized_data = clipping_max_norm(model_sum_attn_attr, cutoff_val=saved_model_cutoffs[model_name])
    residue_com_map[n] = normalized_data

mean_residue_com_map = np.mean(residue_com_map, axis=0)

100%|██████████| 21/21 [00:03<00:00,  5.45it/s]


In [22]:
score = "fitness_score"
score_noC = "fitness_score_nocoupling"

E_array = np.zeros(N_SEQ)
E_nocoupling_array = np.zeros(N_SEQ)
for i in range(N_SEQ):
    seq = AMP_aligned_sequences[i]
    com_map = mean_residue_com_map[i]
    E_array[i] = predict_E(seq, com_map, h_map, epistatic_map, resno_array)
    E_nocoupling_array[i] = predict_E(seq, com_map, h_map, epistatic_map, resno_array, coupling=False)

AMP_df[score] = E_array
AMP_df[score_noC] = E_nocoupling_array

delta_pic50_array = []
delta_score_array = []
delta_score_noC_array = []

for name in AMP_df["Virus_name"].unique():
    virus_df = AMP_df[AMP_df["Virus_name"] == name]
    
    sens_wt_pic50 = virus_df.loc[virus_df["Mutations"].str.contains("sensitive-WT"), "pIC50"].values[0]
    sens_wt_score = virus_df.loc[virus_df["Mutations"].str.contains("sensitive-WT"), score].values[0]
    sens_wt_score_noC = virus_df.loc[virus_df["Mutations"].str.contains("sensitive-WT"), score_noC].values[0]
    
    res_wt_pic50 = virus_df.loc[virus_df["Mutations"].str.contains("resistant-WT"), "pIC50"].values[0]
    res_wt_score = virus_df.loc[virus_df["Mutations"].str.contains("resistant-WT"), score].values[0]
    res_wt_score_noC = virus_df.loc[virus_df["Mutations"].str.contains("resistant-WT"), score_noC].values[0]
    
    delta_pic50_array.append(sens_wt_pic50 - res_wt_pic50)
    delta_score_array.append(sens_wt_score - res_wt_score)
    delta_score_noC_array.append(sens_wt_score_noC - res_wt_score_noC)

    rev_pic50 = virus_df.loc[~virus_df["Mutations"].str.contains("WT"), "pIC50"].values
    rev_score = virus_df.loc[~virus_df["Mutations"].str.contains("WT"), score].values
    rev_score_noC = virus_df.loc[~virus_df["Mutations"].str.contains("WT"), score_noC].values
    
    delta_pic50_array.extend(sens_wt_pic50 - rev_pic50)
    delta_score_array.extend(sens_wt_score - rev_score)
    delta_score_noC_array.extend(sens_wt_score_noC - rev_score_noC)

print("With Coupling Term:", spearmanr(delta_pic50_array, delta_score_array))
print("Without Coupling Term:", spearmanr(delta_pic50_array, delta_score_noC_array))

With Coupling Term: SpearmanrResult(correlation=0.8777106183897393, pvalue=1.7045100488483285e-05)
Without Coupling Term: SpearmanrResult(correlation=0.7356222191012364, pvalue=0.0017734459684828144)


In [23]:
def get_node_coupling(target_node, sequence, communication_map, residue_index, pval_cutoff=1.0, count_cutoff=0):
    """
    Extracts the detailed energetic components for a specific target node.
    Assumes `h_map` and `epistatic_map` are available in the global scope.
    """
    target_reslabel = target_node[0]
    target_idx = np.where(residue_index == target_reslabel)[0][0]
    target_aa = sequence[target_idx]
    L = len(sequence)
    node_inf = np.zeros(L)
    node_c = np.zeros(L)
    node_H = h_map.get((target_reslabel, target_aa), 0.0)
    node_comm = communication_map[target_idx]
    for src_idx in range(L):
        w = node_comm[src_idx]
        if w == 0.0:
            continue
        src_reslabel = residue_index[src_idx]
        src_aa = sequence[src_idx]
        ep_data = epistatic_map.get((target_reslabel, target_aa, src_reslabel, src_aa))
        if ep_data is not None:
            c, _, p_val, s_count = ep_data
            if p_val < pval_cutoff and s_count > count_cutoff:
                node_c[src_idx] = c
                node_inf[src_idx] = w * c
    node_E = node_H + node_inf.sum()
    return node_E, node_H, node_inf, node_comm, node_c


def find_top_coupling(node_inf, residue_index, top_k=15):
    """
    Finds the indices of the top contributing nodes based on coupling influence.
    """
    abs_inf = np.abs(node_inf)
    top_indices = np.argsort(abs_inf)[::-1][:top_k]
    return top_indices

In [35]:
name = "V703_0790"
target_node = ("281", "T")

print(f"------- Target node: {target_node} -------")
target_idx = np.where(resno_array == target_node[0])[0][0]
df = AMP_df[AMP_df["Virus_name"] == name]

print(name, "Sensitive-WT")
sens_seqno = df["Seq_no"][df["Mutations"] == "sensitive-WT"].values[0]
sens_ndx = np.where(AMP_seqno_array == sens_seqno)[0][0]
sens_seq = AMP_aligned_sequences[sens_ndx]
sens_com_map = mean_residue_com_map[sens_ndx]

sens_E_score, sens_H, sens_C = predict_E(sens_seq, sens_com_map, h_map, epistatic_map, resno_array, return_map=True)
sens_node_E, sens_node_H, sens_node_inf, sens_node_comm, sens_node_c = get_node_coupling(target_node, sens_seq, sens_com_map, resno_array)
assert sens_H[target_idx] == sens_node_H and np.isclose(sens_C[target_idx], sens_node_inf.sum())
sens_top_resndx = find_top_coupling(sens_node_inf, resno_array, top_k=8)
print(sens_node_H, sens_node_inf.sum(), sens_node_H+sens_node_inf.sum())
print(list(resno_array[sens_top_resndx] + sens_seq[sens_top_resndx]))

print("\n")
print(name, "Resistant-WT")
res_seqno = df["Seq_no"][df["Mutations"] == "resistant-WT"].values[0]
res_ndx = np.where(AMP_seqno_array == res_seqno)[0][0]
res_seq = AMP_aligned_sequences[res_ndx]
res_com_map = mean_residue_com_map[res_ndx]

res_E_score, res_H, res_C = predict_E(res_seq, res_com_map, h_map, epistatic_map, resno_array, return_map=True)
res_node_E, res_node_H, res_node_inf, res_node_comm, res_node_c = get_node_coupling(target_node, res_seq, res_com_map, resno_array)
assert res_H[target_idx] == res_node_H and np.isclose(res_C[target_idx], res_node_inf.sum())
res_top_resndx = find_top_coupling(res_node_inf, resno_array, top_k=8)
print(res_node_H, res_node_inf.sum(), res_node_H+res_node_inf.sum())
print(list(resno_array[res_top_resndx] + res_seq[res_top_resndx]))

------- Target node: ('281', 'T') -------
V703_0790 Sensitive-WT
-0.37411554604768754 -0.05019547795616611 -0.4243110240038537
['283T', '280N', '236T', '318F', '277L', '276O', '279D', '321G']


V703_0790 Resistant-WT
-0.37411554604768754 -0.06559436410469241 -0.4397099101523799
['283T', '236T', '318F', '277L', '276O', '279D', '280S', '321G']


In [4]:
def find_differences(sensitive_wt, resistant_wt):
    if len(sensitive_wt) != len(resistant_wt):
        raise ValueError("Sequences must be of the exact same length.")
    differences = []
    for i in range(len(sensitive_wt)):
        if sensitive_wt[i] != resistant_wt[i]:
            differences.append({
                'position': i,
                'sensitive_char': sensitive_wt[i],
                'resistant_char': resistant_wt[i]
            })
    return differences

def generate_intermediate_states(sensitive_wt, differences):
    options = [(diff['sensitive_char'], diff['resistant_char']) for diff in differences]
    all_combinations = list(itertools.product(*options))
    intermediate_sequences = []
    for combo in all_combinations:
        seq_list = list(sensitive_wt)
        for idx, diff in enumerate(differences):
            seq_list[diff['position']] = combo[idx]
        intermediate_sequences.append("".join(seq_list))
    return intermediate_sequences

In [63]:
name = "V703_1714" #["V703_0790", "V703_1586", "V703_1714"]
print("------", name, "------")

traj_dict = {
    "Sequence": [],
    "E_score_sens": [], "n_mutations_sens": [],
    "E_score_res": [], "n_mutations_res": []
}
df = AMP_df[AMP_df["Virus_name"] == name]

sens_seq = np.array(list(df["Sequence"][df["Mutations"] == "sensitive-WT"].values[0]))
sens_seqno = df["Seq_no"][df["Mutations"] == "sensitive-WT"].values[0]
sens_ndx = np.where(AMP_seqno_array == sens_seqno)[0][0]
sens_com_map = mean_residue_com_map[sens_ndx]

res_seq = np.array(list(df["Sequence"][df["Mutations"] == "resistant-WT"].values[0]))
res_seqno = df["Seq_no"][df["Mutations"] == "resistant-WT"].values[0]
res_ndx = np.where(AMP_seqno_array == res_seqno)[0][0]
res_com_map = mean_residue_com_map[res_ndx]

differences = find_differences(sens_seq, res_seq)
states = generate_intermediate_states(sens_seq, differences)
print(f"Mutations found: {len(differences)} (from sensitive to resistant)")
for d in differences:
    print(f" - Residue {resno_array[d['position']]}: {d['sensitive_char']} -> {d['resistant_char']}")
print(f"All possible states ({len(states)} total)")

for s, state in enumerate(states):
    seq = np.array(list(state))
    traj_dict["Sequence"].append("".join(seq))
    
    n_mutations_sens = (sens_seq != seq).sum()
    if n_mutations_sens > 0:
        E_score_sens = predict_E(seq, sens_com_map, h_map, epistatic_map, resno_array)
    else:
        E_score_sens = np.nan
    traj_dict["n_mutations_sens"].append(n_mutations_sens)
    traj_dict["E_score_sens"].append(E_score_sens)

    n_mutations_res = (res_seq != seq).sum()
    if n_mutations_res > 0:
        E_score_res = predict_E(seq, res_com_map, h_map, epistatic_map, resno_array)
    else:
        E_score_res = np.nan
    traj_dict["n_mutations_res"].append(n_mutations_res)
    traj_dict["E_score_res"].append(E_score_res)

traj_df = pd.DataFrame(traj_dict)
traj_df["Seq_no"] = traj_df.index + 1
#traj_df.to_csv("../data/"+name+"_trajectory.csv", index=False)

------ V703_1714 ------
Mutations found: 8 (from sensitive to resistant)
 - Residue 135: O -> T
 - Residue 321: - -> N
 - Residue 332: N -> O
 - Residue 334: O -> S
 - Residue 392: N -> T
 - Residue 459: G -> D
 - Residue 462: D -> E
 - Residue 644: T -> K
All possible states (256 total)


In [4]:
name = "V703_1714"
print("Processing", name, "...")
traj_df = pd.read_csv("../data/"+name+"_trajectory.csv")
traj_seqno_array = traj_df["Seq_no"].values
traj_aligned_sequences = np.array([list(seq) for seq in traj_df['Sequence']])
N_SEQ, L_SEQ = traj_aligned_sequences.shape

#run src/run_attentions.py to get a raw attention map for each sequences
attn_dir = "../results/AMP/attention/"+name

residue_com_map = np.zeros([len(selected_models), N_SEQ, L_SEQ, L_SEQ])
for n, model_name in enumerate(selected_models):
    model_sum_attn_attr = np.zeros([N_SEQ, L_SEQ, L_SEQ])
    attn_file_paths = [os.path.join(attn_dir, model_name, f"{no}_attentions_raw.npy") for no in traj_df["Seq_no"]]

    with ThreadPoolExecutor(max_workers=8) as executor: 
        attn_results = list(tqdm(executor.map(process_single_attention, attn_file_paths), total=N_SEQ))
    
    for i, attn_res in enumerate(attn_results):
        model_sum_attn_attr[i] = attn_res
    
    normalized_data = clipping_max_norm(model_sum_attn_attr, cutoff_val=saved_model_cutoffs[model_name])
    residue_com_map[n] = normalized_data

traj_mean_residue_com_map = np.mean(residue_com_map, axis=0)

  0%|          | 0/256 [00:00<?, ?it/s]

Processing V703_1714 ...


100%|██████████| 256/256 [00:51<00:00,  4.95it/s]


In [6]:
print("------", name, "------")
E_array = np.zeros(N_SEQ)
for i in tqdm(range(N_SEQ)):
    seq = traj_aligned_sequences[i]
    com_map = traj_mean_residue_com_map[i]
    E_array[i] = predict_E(seq, com_map, h_map, epistatic_map, resno_array)
traj_df["fitness_score"] = E_array

  1%|          | 3/256 [00:00<00:10, 23.30it/s]

------ V703_1714 ------


100%|██████████| 256/256 [00:09<00:00, 27.31it/s]


In [18]:
top_k = 3
max_n_mut = 3 #traj_df["n_mutations_sens"].max()
for n_mut in range(1, max_n_mut + 1):
    print("> Trajectory: Sensitive-Resistant")
    sens_sub_traj = traj_df[(traj_df["n_mutations_sens"] == n_mut)]
    sens_sub_traj = sens_sub_traj.sort_values(by=["fitness_score"])
    sens_inferred_E = sens_sub_traj["E_score_sens"].values
    sens_E = sens_sub_traj["fitness_score"].values
    print(f"Number of mutations: {n_mut} (n={len(sens_sub_traj)})")
    print(f"MSE={mean_squared_error(sens_E, sens_inferred_E)}")

    wt_seq = np.array(list(traj_df[(traj_df["n_mutations_sens"] == 0)]["Sequence"].values[0]))
    wt_score = traj_df[(traj_df["n_mutations_sens"] == 0)]["fitness_score"].values[0]
    print("-- Top states --")
    for seq, score in zip(sens_sub_traj.head(top_k)["Sequence"], sens_sub_traj.head(top_k)["fitness_score"]):
        seq = np.array(list(seq))
        mutations = []
        for pos in np.where(wt_seq != seq)[0]:
            mut_res = resno_array[pos]
            mut_aa = seq[pos]
            wt_aa = wt_seq[pos]
            mutations.append(wt_aa+mut_res+mut_aa)
        print("Mutations:", mutations)
        print(f"delta E = {score-wt_score}")
    
    print("\n> Trajectory: Resistant-Sensitive")
    res_sub_traj = traj_df[(traj_df["n_mutations_res"] == n_mut)]
    res_sub_traj = res_sub_traj.sort_values(by=["fitness_score"], ascending=False)
    res_inferred_E = res_sub_traj["E_score_res"].values
    res_E = res_sub_traj["fitness_score"].values
    print(f"Number of mutations: {n_mut} (n={len(res_sub_traj)})")
    print(f"MSE={mean_squared_error(res_E, res_inferred_E)}")

    wt_seq = np.array(list(traj_df[(traj_df["n_mutations_res"] == 0)]["Sequence"].values[0]))
    wt_score = traj_df[(traj_df["n_mutations_res"] == 0)]["fitness_score"].values[0]
    print("-- Top states --")
    for seq, score in zip(res_sub_traj.head(top_k)["Sequence"], res_sub_traj.head(top_k)["fitness_score"]):
        seq = np.array(list(seq))
        mutations = []
        for pos in np.where(wt_seq != seq)[0]:
            mut_res = resno_array[pos]
            mut_aa = seq[pos]
            wt_aa = wt_seq[pos]
            mutations.append(wt_aa+mut_res+mut_aa)
        print("Mutations:", mutations)
        print(f"delta E = {score-wt_score}")

    inferred_E = np.concatenate([sens_inferred_E, res_inferred_E])
    actual_E = np.concatenate([sens_E, res_E])
    print(f"\n>>> MSE={mean_squared_error(actual_E, inferred_E)} <<<")
    print("-------------")

> Trajectory: Sensitive-Resistant
Number of mutations: 1 (n=8)
MSE=0.016246641807893744
-- Top states --
Mutations: ['N332O']
delta E = -0.7428432104829863
Mutations: ['T644K']
delta E = -0.5741117208274513
Mutations: ['G459D']
delta E = -0.3336451596980172

> Trajectory: Resistant-Sensitive
Number of mutations: 1 (n=8)
MSE=0.012161528815936583
-- Top states --
Mutations: ['O332N']
delta E = 0.6637603113913393
Mutations: ['K644T']
delta E = 0.511603854320068
Mutations: ['T135O']
delta E = 0.48125090803743653

>>> MSE=0.014204085311915166 <<<
-------------
> Trajectory: Sensitive-Resistant
Number of mutations: 2 (n=28)
MSE=0.02440529964842712
-- Top states --
Mutations: ['N332O', 'T644K']
delta E = -1.2485549360036288
Mutations: ['N332O', 'G459D']
delta E = -1.0751695694290735
Mutations: ['G459D', 'T644K']
delta E = -0.9857350312311386

> Trajectory: Resistant-Sensitive
Number of mutations: 2 (n=28)
MSE=0.026708665396134673
-- Top states --
Mutations: ['O332N', 'K644T']
delta E = 1.2046

In [ ]:
# sensitive, resistant
AMP_seqno = {
    "V703_0790": (2166, 2167),
    "V703_1714": (2155, 2156),
    "V703_2769": (2088, 2087),
    "V703_1586": (2175, 2176),
    "V703_0510": (2173, 2174),
    "V703_0514": (2061, 2062),
}